In [2]:
import sys 
from math import floor, ceil

sys.path.append('../src')
import sys
print(sys.executable)
from main import DataPipeline

pipeline = DataPipeline()
df = pipeline.pre_process_data()
df_engineered = pipeline.feature_engineering(df)
df_engineered.columns

/Users/jordanpickles/opt/miniconda3/envs/whoop-pipeline/bin/python


Index(['recovery_score', 'resting_heart_rate', 'hrv_rmssd_milli',
       'skin_temp_celsius', 'sleep_efficiency_percentage',
       'sleep_consistency_percentage', 'sleep_performance_percentage',
       'respiratory_rate', 'sleep_cycle_count', 'disturbance_count',
       'sleep_needed_baseline_milli',
       'sleep_needed_need_from_recent_strain_milli', 'cycle_strain',
       'cycle_avg_heart_rate', 'cycle_kilojoule', 'total_in_bed_time_hours',
       'total_rem_sleep_time_hours', 'total_sleep_time_hours',
       'hrv_rmssd_milli_rolling_avg_7', 'resting_heart_rate_rolling_avg_7',
       'cycle_strain_rolling_avg_7', 'day_of_week', 'anomalous_day_flag',
       'sleep_start_local_decimal', 'sleep_end_local_decimal',
       'recovery_score_shift'],
      dtype='object')

# 1.1 Splitting the data
Here a 70/30 split in a temporally ascending order will be taken with a 7 day washout period between these splits to ensure there is data leakage with the 7 day rolling averages.
70:30 split used in this time series dataset to capture any seasonal trends within the data

In [3]:
row_count = len(df)
row_count_washout_removal = row_count - 7
train_upper_bound = floor(row_count_washout_removal * 0.7)
test_lower_bound = train_upper_bound + 7 
df = df.sort_values('date_local').reset_index(drop=True)
train = df.iloc[:train_upper_bound]
test = df.iloc[test_lower_bound:]
print(f"Train - Start Date: {train['date_local'].min()} | End Date: {train['date_local'].max()} | Number of Rows: {len(train)} | % of Total Rows: {len(train) / row_count}")
print(f"Test - Start Date: {test['date_local'].min()} | End Date: {test['date_local'].max()} | Number of Rows: {len(test)} | % of Total Rows: {len(test) / row_count}")


Train - Start Date: 2024-01-01 06:30:35.725000 | End Date: 2025-09-11 03:42:40.847000 | Number of Rows: 613 | % of Total Rows: 0.6942242355605889
Test - Start Date: 2025-09-20 08:01:07.809000 | End Date: 2026-06-09 05:17:26.999000 | Number of Rows: 263 | % of Total Rows: 0.29784824462061155


# 1.2 Split out Features and Label
Labels: Recovery Score Shift (-1 day)

In [4]:
features = [col for col in train.columns if col not in ['recovery_score_shift', 'date_local']]
print(features)

X_train = train[features]
Y_train = train['recovery_score_shift']

X_test = test[features]
Y_test = test['recovery_score_shift']

['cycle_id', 'date', 'recovery_score', 'resting_heart_rate', 'hrv_rmssd_milli', 'spo2_percentage', 'skin_temp_celsius', 'sleep_start', 'sleep_end', 'timezone_offset', 'sleep_efficiency_percentage', 'sleep_consistency_percentage', 'sleep_performance_percentage', 'respiratory_rate', 'sleep_cycle_count', 'disturbance_count', 'sleep_needed_baseline_milli', 'sleep_needed_need_from_recent_strain_milli', 'cycle_strain', 'cycle_avg_heart_rate', 'cycle_max_heart_rate', 'cycle_kilojoule', 'total_in_bed_time_hours', 'total_awake_time_hours', 'total_light_sleep_time_hours', 'total_slow_wave_sleep_time_hours', 'total_rem_sleep_time_hours', 'total_sleep_time_hours', 'sleep_start_local', 'sleep_end_local']


KeyError: 'recovery_score_shift'